# 7-СС · Алты есептеу есебі

## 1-есеп
**Берілгені:** белгілеуде нысанның 30 пикселі, болжамда 25, ортақ 20. IoU мен Dice-ті табыңыз.

Біріктіру (union) = 30 + 25 − 20 = 35

- IoU = 20 / 35 
- Dice = 2·20 / (30 + 25) = 40 / 55 

In [1]:
gt, pred, inter = 30, 25, 20
union = gt + pred - inter
iou = inter / union
dice = 2 * inter / (gt + pred)
print(f"Union = {union}")
print(f"IoU  = {iou:.3f}")
print(f"Dice = {dice:.3f}")

Union = 35
IoU  = 0.571
Dice = 0.727


## 2-есеп
**Берілгені:** IoU = 0,6. Dice неге тең? Dice IoU-дан кіші бола ала ма?

Байланыс формуласы: Dice = 2·IoU / (1 + IoU)


In [2]:
iou = 0.6
dice = 2 * iou / (1 + iou)
print(f"Dice = {dice:.2f}")

# Dice >= IoU екенін тексеру
import numpy as np
ious = np.linspace(0, 1, 101)
dices = 2 * ious / (1 + ious)
print("Dice >= IoU әрқашан орындала ма:", bool(np.all(dices >= ious - 1e-12)))

Dice = 0.75
Dice >= IoU әрқашан орындала ма: True


## 3-есеп
**Берілгені:** FCN-32s кірісі — 512×512. Үлкейтуге дейінгі баға картасының өлшемі қандай?

FCN-32s кескінді 32 есе кішірейтеді (stride = 32).

512 / 32 = 16

In [3]:
size, stride = 512, 32
print(f"Баға картасы: {size // stride}×{size // stride}")

Баға картасы: 16×16


## 4-есеп
**Берілгені:** транспонирленген конволюция: I = 16, k = 4, s = 2, p = 1. Шығыс өлшемі?

Формула: O = (I − 1)·s − 2p + k


In [4]:
I, k, s, p = 16, 4, 2, 1
O = (I - 1) * s - 2 * p + k
print(f"Шығыс өлшемі: {O}")

# PyTorch-пен тексеру
import torch
import torch.nn as nn
x = torch.zeros(1, 1, I, I)
y = nn.ConvTranspose2d(1, 1, kernel_size=k, stride=s, padding=p)(x)
print("PyTorch:", tuple(y.shape))

Шығыс өлшемі: 32
PyTorch: (1, 1, 32, 32)


## 5-есеп
**Берілгені:** Focal loss, γ = 2, α-сыз: p_t = 0,8 пикселінің шығыны. CE-ден неше есе кіші?

- CE = −log(p_t) = −log(0.8)
- Focal loss = (1 − p_t)^γ · CE 


In [5]:
import math
p_t, gamma = 0.8, 2
ce = -math.log(p_t)
fl = (1 - p_t) ** gamma * ce
print(f"CE          = {ce:.4f}")
print(f"Focal loss  = {fl:.4f}")
print(f"CE / FL     = {ce / fl:.1f} есе")

CE          = 0.2231
Focal loss  = 0.0089
CE / FL     = 25.0 есе


## 6-есеп
**Берілгені:** ісік — суреттің 2 % пикселі. Қандай шығын функциясын таңдау керек және неге pixel accuracy емес?

**Неге pixel accuracy жарамайды.** Модель барлық пикселді «фон» деп болжаса, ол 98 % дәлдік алады, бірақ ісікті мүлде таппайды. Яғни метрика класстардың теңсіздігін ескермейді және жалған жоғары нәтиже береді.

**Не таңдау керек.**
- **Dice loss** (немесе IoU loss): тек нысанның қабаттасуын өлшейді, фонның көптігіне тәуелді емес.
- **Focal loss** немесе салмақталған CE: сирек класқа көбірек салмақ береді.
- Тәжірибеде жиі **CE + Dice** комбинациясын қолданады.

**Бағалау үшін** Dice немесе IoU қолданамыз.

In [6]:
import numpy as np
# 100x100 сурет, ісік = 2% пиксель
true = np.zeros((100, 100), dtype=int)
true[:10, :20] = 1          # 200 пиксель = 2%
pred = np.zeros_like(true)  # модель бәрін "фон" дейді

acc = (pred == true).mean()
inter = np.logical_and(pred == 1, true == 1).sum()
dice = 2 * inter / ((pred == 1).sum() + (true == 1).sum() + 1e-6)
print(f"Ісік үлесі     : {true.mean():.0%}")
print(f"Pixel accuracy : {acc:.0%}")
print(f"Dice           : {dice:.2f}")

Ісік үлесі     : 2%
Pixel accuracy : 98%
Dice           : 0.00
